# Практическая работа №2: Обработка выборочных данных. Нахождение точечных оценок параметров распределения

Выполнили студентки гр. 2384 Лавренова Юлия и Соц Екатерина. Вариант №32

## Цель работы
Получение практических навыков нахождения точечных статистических оценок параметров распределения.

## Основные теоретические положения
**Точечная оценка** - это приближенное значение неизвестного параметра генеральной совокупности, вычисленное по выборочным данным. Надежная точечная оценка должна удовлетворять свойствам:
- **Несмещенность** - математическое ожидание оценки равно оцениваемому параметру;
- **Состоятельность** - оценка сходится по вероятность к параметру при увеличении объема выборки;
- **Эффективность** - оценка имеет наименьшую дисперсию среди всех оценок.

**Начальный эмпирический момент k-го порядка:**
$\bar{M_k} = \frac{1}{N}\sum n_jx_j^k$, где

$N$ - объем выборки,

$n_j$ - абсолютная частота,

$x_j$ - середина интервала.


**Центральный эмпирический момент k-ого порядка:**
$\bar{m_k} = \frac{1}{N}\sum n_j(x_j - \bar{x_в})^k$,

где $\bar{x_в} = \bar{M_1} = \frac{1}{N}\sum n_jx_j$.

**Выборочная дисперсия**: $D_в = \bar{m_2} = \frac{1}{N}\sum n_j(x_j - \bar{x_в})^2$


**Исправленная оценка дисперсии:** $s^2 = \frac{N}{N-1}D_в$

**Статистическая оценка СКО:** $\sigma_в = \sqrt D_в$

**Статистическая оценка асимметрии:** $\bar{A_s} = \frac{\bar{m_3}}{s^3}$

**Статистическая оценка эксцесса:** $\bar{E} = \frac{\bar{m_4}}{s^4} - 3$

**Метод условных вариант** используется для упрощения вычислений при обработке интервальных рядов. Суть метода заключается в переходе к новым переменным - условным вариантам:
$u_j=\frac{x_j - C}{h}$ , где:

$x_j$ — середина j-го интервала

$C$ — условный нуль

$h$ — шаг

**Условный момент к-ого порядка:** $\bar{M_k}^* = \frac{1}{N}\sum n_j (\frac{x_j - C}{h})^k = \frac{1}{N}\sum n_j u_j^k$

### Робастные выборочные статистические оценки:
1. Модой вариационного ряда называется варианта, которой соответсвует наибольшая частота. $M_o^* = x_{i-1} + h \frac{\tilde{m_i} - \tilde{m_{i-1}}}{(\tilde{m_i} - \tilde{m_{i-1}}) + (\tilde{m_i} - \tilde{m_{i+1}})}$, где $x_{i-1}$ - левая граница интервала, $h$ - длина интервала в интервальном ряде, $\tilde{m_i}$ - относительная частота для i-ого интервала.
2. Медианой называется варианта, соответсвующая середине ранжированного ряда. $M_e^* = x_{i-1} + \frac{h}{\tilde{m_i}}(0.5 - \sum_{j=1}^{i-1} \tilde{m_j})$
3. Выборочный коэффициент вариации характеризует разброс значений вариант относительно выборочного среднего: $V^* = \frac{\sigma_в}{\bar{x_в}} \cdot 100 \%$

## Постановка задачи
Для заданных выборочных данных вычислить с использованием метода моментов и условных вариант точечные статистические оценки математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации исследуемой случайной величины. Полученные результаты содержательно проинтерпретировать.

## Выполнение работы

### 1. Для середин интервального ряда, полученного в практической работе №1, вычислить условные варианты. Заполнить таблицу (в последней строке $\sum$ необходимо заполнить суммы столбцов; ячейки отмеченные прочерком заполнять не надо). Провести контроль вычислений.

Наработки из первой практической работы:

In [1]:
import pandas as pd

data = pd.read_csv("sample_wine_data.csv")

In [2]:
import numpy as np

def group_array(data):
  # ранжированный
  df_range = np.sort(data)

  # вариационный
  df_unique = np.unique(df_range)
  df_var = {value: np.sum(df_range == value) for value in df_unique}

  # интервальный

  N = 114
  k = round(1 + 3.322*np.log10(N))
  h = (max(df_unique) - min(df_unique)) / k
  bins = np.linspace(min(df_unique), max(df_unique), k)
  m_i, edges = np.histogram(data, bins=bins)

  x_i_mid = (edges[:-1] + edges[1:]) / 2
  m_i_rel = m_i / N
  m_i_cum = np.cumsum(m_i)
  m_i_rel_cum = np.cumsum(m_i_rel)
  interval_series = pd.DataFrame({
      "Интервал": [f"{round(edges[i],2)} – {round(edges[i+1],2)}" for i in range(len(m_i))],
      "Абсолютная частота": m_i,
      "Относительная частота": m_i / N
  })
  table = pd.DataFrame({
      "i": range(1, len(m_i)+1),
      "$[x_i, x_{i+1})$": [f"[{edges[i]:.2f}, {edges[i+1]:.2f})" for i in range(len(m_i))],
      "$\\tilde{x}_i$": np.round(x_i_mid, 3),
      "$m_i$": m_i,
      "$\\tilde{m}_i$": np.round(m_i_rel, 4),
      "$m_{i}^{нак}$": m_i_cum,
      "$\\tilde{m}_{i}^{нак}$": np.round(m_i_rel_cum, 4)
  })


  sum_row = pd.DataFrame({
      "i": ["Σ"],
      "$[x_i, x_{i+1})$": ["-"],
      "$\\tilde{x}_i$": ["-"],
      "$m_i$": [m_i.sum()],
      "$\\tilde{m}_i$": [m_i_rel.sum()],
      "$m_{i}^{нак}$": ["-"],
      "$\\tilde{m}_{i}^{нак}$": ["-"]
  })

  table = pd.concat([table, sum_row], ignore_index=True)

  return df_range, df_var, interval_series, table
data_ph = np.array(data["pH"])
ph_range, ph_var, ph_interval, ph_table = group_array(data_ph)

from IPython.display import display, Markdown

display(Markdown(ph_table.to_markdown(index=False)))

data_sulphates = np.array(data["sulphates"])
sulphates_range, sulphates_var, sulphates_interval, sulphates_table = group_array(data_sulphates)

display(Markdown(sulphates_table.to_markdown(index=False)))

| i   | $[x_i, x_{i+1})$   | $\tilde{x}_i$   |   $m_i$ |   $\tilde{m}_i$ | $m_{i}^{нак}$   | $\tilde{m}_{i}^{нак}$   |
|:----|:-------------------|:----------------|--------:|----------------:|:----------------|:------------------------|
| 1   | [2.86, 3.01)       | 2.934           |       5 |          0.0439 | 5               | 0.0439                  |
| 2   | [3.01, 3.16)       | 3.083           |      12 |          0.1053 | 17              | 0.1491                  |
| 3   | [3.16, 3.31)       | 3.231           |      38 |          0.3333 | 55              | 0.4825                  |
| 4   | [3.31, 3.45)       | 3.38            |      41 |          0.3596 | 96              | 0.8421                  |
| 5   | [3.45, 3.60)       | 3.529           |      13 |          0.114  | 109             | 0.9561                  |
| 6   | [3.60, 3.75)       | 3.677           |       3 |          0.0263 | 112             | 0.9825                  |
| 7   | [3.75, 3.90)       | 3.826           |       2 |          0.0175 | 114             | 1.0                     |
| Σ   | -                  | -               |     114 |          1      | -               | -                       |

| i   | $[x_i, x_{i+1})$   | $\tilde{x}_i$   |   $m_i$ |   $\tilde{m}_i$ | $m_{i}^{нак}$   | $\tilde{m}_{i}^{нак}$   |
|:----|:-------------------|:----------------|--------:|----------------:|:----------------|:------------------------|
| 1   | [0.44, 0.61)       | 0.524           |      53 |          0.4649 | 53              | 0.4649                  |
| 2   | [0.61, 0.78)       | 0.693           |      42 |          0.3684 | 95              | 0.8333                  |
| 3   | [0.78, 0.95)       | 0.861           |      13 |          0.114  | 108             | 0.9474                  |
| 4   | [0.95, 1.11)       | 1.03            |       1 |          0.0088 | 109             | 0.9561                  |
| 5   | [1.11, 1.28)       | 1.199           |       2 |          0.0175 | 111             | 0.9737                  |
| 6   | [1.28, 1.45)       | 1.367           |       2 |          0.0175 | 113             | 0.9912                  |
| 7   | [1.45, 1.62)       | 1.536           |       1 |          0.0088 | 114             | 1.0                     |
| Σ   | -                  | -               |     114 |          1      | -               | -                       |

In [3]:
# Извлекаем данные из ph_table
ph_data = ph_table.iloc[:-1].copy()

ph_data['i'] = ph_data['i'].astype(int)
ph_data['$\\tilde{x}_i$'] = ph_data['$\\tilde{x}_i$'].astype(float)
ph_data['$m_i$'] = ph_data['$m_i$'].astype(int)

print("Исходные данные для pH:")
display(Markdown(ph_data[['i', '$\\tilde{x}_i$', '$m_i$']].to_markdown(index=False)))

Исходные данные для pH:


|   i |   $\tilde{x}_i$ |   $m_i$ |
|----:|----------------:|--------:|
|   1 |           2.934 |       5 |
|   2 |           3.083 |      12 |
|   3 |           3.231 |      38 |
|   4 |           3.38  |      41 |
|   5 |           3.529 |      13 |
|   6 |           3.677 |       3 |
|   7 |           3.826 |       2 |

In [4]:
def calculate_conditional_variants(data_table, feature_name):

    df = data_table.copy()

    # Определяем шаг интервала h
    # Берем разницу между первыми двумя серединами интервалов
    x_mid = df['$\\tilde{x}_i$'].values
    h = round(x_mid[1] - x_mid[0], 3)

    # Выбираем условный нуль C (середина интервала с максимальной частотой)
    max_freq_idx = df['$m_i$'].idxmax()
    C = df.loc[max_freq_idx, '$\\tilde{x}_i$']

    print(f"ВЫЧИСЛЕНИЕ УСЛОВНЫХ ВАРИАНТ ДЛЯ ПРИЗНАКА: {feature_name}")
    print(f"Шаг интервала h = {h}")
    print(f"Условный нуль C = {C} (середина интервала с максимальной частотой)")

    # условные варианты u_i
    df['$u_i$'] = ((df['$\\tilde{x}_i$'] - C) / h).round(3)

    df['$n_i u_i$'] = (df['$m_i$'] * df['$u_i$']).round(3)
    df['$n_i u_i^2$'] = (df['$m_i$'] * df['$u_i$']**2).round(3)
    df['$n_i u_i^3$'] = (df['$m_i$'] * df['$u_i$']**3).round(3)
    df['$n_i u_i^4$'] = (df['$m_i$'] * df['$u_i$']**4).round(3)
    df['$n_i (u_i+1)^4$'] = (df['$m_i$'] * (df['$u_i$'] + 1)**4).round(3)

    result_table = df[['i', '$\\tilde{x}_i$', '$m_i$', '$u_i$',
                       '$n_i u_i$', '$n_i u_i^2$', '$n_i u_i^3$',
                       '$n_i u_i^4$', '$n_i (u_i+1)^4$']].copy()

    sums = pd.DataFrame({
        'i': ['Σ'],
        '$\\tilde{x}_i$': ['-'],
        '$m_i$': [df['$m_i$'].sum()],
        '$u_i$': ['-'],
        '$n_i u_i$': [result_table['$n_i u_i$'].sum()],
        '$n_i u_i^2$': [result_table['$n_i u_i^2$'].sum()],
        '$n_i u_i^3$': [result_table['$n_i u_i^3$'].sum()],
        '$n_i u_i^4$': [result_table['$n_i u_i^4$'].sum()],
        '$n_i (u_i+1)^4$': [result_table['$n_i (u_i+1)^4$'].sum()]
    })

    final_table = pd.concat([result_table, sums], ignore_index=True)

    return final_table, h, C, df['$m_i$'].sum()

# для pH
ph_conditional_table, ph_h, ph_C, ph_n = calculate_conditional_variants(ph_data, "pH")

display(Markdown(ph_conditional_table.to_markdown(index=False)))

ВЫЧИСЛЕНИЕ УСЛОВНЫХ ВАРИАНТ ДЛЯ ПРИЗНАКА: pH
Шаг интервала h = 0.149
Условный нуль C = 3.38 (середина интервала с максимальной частотой)


| i   | $\tilde{x}_i$   |   $m_i$ | $u_i$   |   $n_i u_i$ |   $n_i u_i^2$ |   $n_i u_i^3$ |   $n_i u_i^4$ |   $n_i (u_i+1)^4$ |
|:----|:----------------|--------:|:--------|------------:|--------------:|--------------:|--------------:|------------------:|
| 1   | 2.934           |       5 | -2.993  |     -14.965 |        44.79  |      -134.057 |       401.233 |            78.886 |
| 2   | 3.083           |      12 | -1.993  |     -23.916 |        47.665 |       -94.996 |       189.326 |            11.668 |
| 3   | 3.231           |      38 | -1.0    |     -38     |        38     |       -38     |        38     |             0     |
| 4   | 3.38            |      41 | 0.0     |       0     |         0     |         0     |         0     |            41     |
| 5   | 3.529           |      13 | 1.0     |      13     |        13     |        13     |        13     |           208     |
| 6   | 3.677           |       3 | 1.993   |       5.979 |        11.916 |        23.749 |        47.332 |           240.74  |
| 7   | 3.826           |       2 | 2.993   |       5.986 |        17.916 |        53.623 |       160.493 |           508.425 |
| Σ   | -               |     114 | -       |     -51.916 |       173.287 |      -176.681 |       849.384 |          1088.72  |

In [5]:
def check_calc(table, n):
    # Σ n_j (u_j + 1)^4 = Σ n_j u_j^4 + 4Σ n_j u_j^3 + 6Σ n_j u_j^2 + 4Σ n_j u_j + n

    sum_ni_ui = table[table['i'] != 'Σ']['$n_i u_i$'].sum()
    sum_ni_ui2 = table[table['i'] != 'Σ']['$n_i u_i^2$'].sum()
    sum_ni_ui3 = table[table['i'] != 'Σ']['$n_i u_i^3$'].sum()
    sum_ni_ui4 = table[table['i'] != 'Σ']['$n_i u_i^4$'].sum()
    sum_ni_ui_plus1_4 = table[table['i'] != 'Σ']['$n_i (u_i+1)^4$'].sum()

    # Вычисляем правую часть формулы
    right_side = sum_ni_ui4 + 4*sum_ni_ui3 + 6*sum_ni_ui2 + 4*sum_ni_ui + n

    print(f"Σ n_i u_i = {sum_ni_ui:.3f}")
    print(f"Σ n_i u_i^2 = {sum_ni_ui2:.3f}")
    print(f"Σ n_i u_i^3 = {sum_ni_ui3:.3f}")
    print(f"Σ n_i u_i^4 = {sum_ni_ui4:.3f}")
    print(f"Σ n_i (u_i+1)^4 = {sum_ni_ui_plus1_4:.3f}")
    print(f"n = {n}")
    print(f"Левая часть: Σ n_i (u_i+1)^4 = {sum_ni_ui_plus1_4:.3f}")
    print(f"Правая часть: Σ n_i u_i^4 + 4Σ n_i u_i^3 + 6Σ n_i u_i^2 + 4Σ n_i u_i + n =")
    print(f"  {sum_ni_ui4:.3f} + 4*{sum_ni_ui3:.3f} + 6*{sum_ni_ui2:.3f} + 4*{sum_ni_ui:.3f} + {n} =")
    print(f"  {sum_ni_ui4:.3f} + {4*sum_ni_ui3:.3f} + {6*sum_ni_ui2:.3f} + {4*sum_ni_ui:.3f} + {n} =")
    print(f"  {right_side:.3f}")

    if abs(right_side - sum_ni_ui_plus1_4) < 0.01:
        print("\n Равенство выполняется")
        print(f"  Разница: {abs(right_side - sum_ni_ui_plus1_4):.6f}")
    else:
        print("\n Равенство не выполняется")
        print(f"  Разница: {abs(right_side - sum_ni_ui_plus1_4):.6f}")

check_calc(ph_conditional_table, ph_n)

Σ n_i u_i = -51.916
Σ n_i u_i^2 = 173.287
Σ n_i u_i^3 = -176.681
Σ n_i u_i^4 = 849.384
Σ n_i (u_i+1)^4 = 1088.719
n = 114
Левая часть: Σ n_i (u_i+1)^4 = 1088.719
Правая часть: Σ n_i u_i^4 + 4Σ n_i u_i^3 + 6Σ n_i u_i^2 + 4Σ n_i u_i + n =
  849.384 + 4*-176.681 + 6*173.287 + 4*-51.916 + 114 =
  849.384 + -706.724 + 1039.722 + -207.664 + 114 =
  1088.718

 Равенство выполняется
  Разница: 0.001000


In [6]:
# Извлекаем данные для sulphates
sulphates_data = sulphates_table.iloc[:-1].copy()
sulphates_data['i'] = sulphates_data['i'].astype(int)
sulphates_data['$\\tilde{x}_i$'] = sulphates_data['$\\tilde{x}_i$'].astype(float)
sulphates_data['$m_i$'] = sulphates_data['$m_i$'].astype(int)

# Вычисляем условные варианты для sulphates
sulphates_conditional_table, sulphates_h, sulphates_C, sulphates_n = calculate_conditional_variants(sulphates_data, "SULPHATES")

display(Markdown(sulphates_conditional_table.to_markdown(index=False)))

check_calc(sulphates_conditional_table, sulphates_n)

ВЫЧИСЛЕНИЕ УСЛОВНЫХ ВАРИАНТ ДЛЯ ПРИЗНАКА: SULPHATES
Шаг интервала h = 0.169
Условный нуль C = 0.524 (середина интервала с максимальной частотой)


| i   | $\tilde{x}_i$   |   $m_i$ | $u_i$   |   $n_i u_i$ |   $n_i u_i^2$ |   $n_i u_i^3$ |   $n_i u_i^4$ |   $n_i (u_i+1)^4$ |
|:----|:----------------|--------:|:--------|------------:|--------------:|--------------:|--------------:|------------------:|
| 1   | 0.524           |      53 | 0.0     |       0     |         0     |         0     |         0     |            53     |
| 2   | 0.693           |      42 | 1.0     |      42     |        42     |        42     |        42     |           672     |
| 3   | 0.861           |      13 | 1.994   |      25.922 |        51.688 |       103.067 |       205.515 |          1044.6   |
| 4   | 1.03            |       1 | 2.994   |       2.994 |         8.964 |        26.838 |        80.354 |           254.467 |
| 5   | 1.199           |       2 | 3.994   |       7.988 |        31.904 |       127.425 |       508.935 |          1244.01  |
| 6   | 1.367           |       2 | 4.988   |       9.976 |        49.76  |       248.204 |      1238.04  |          2571.33  |
| 7   | 1.536           |       1 | 5.988   |       5.988 |        35.856 |       214.707 |      1285.66  |          2384.58  |
| Σ   | -               |     114 | -       |      94.868 |       220.172 |       762.241 |      3360.51  |          8223.98  |

Σ n_i u_i = 94.868
Σ n_i u_i^2 = 220.172
Σ n_i u_i^3 = 762.241
Σ n_i u_i^4 = 3360.510
Σ n_i (u_i+1)^4 = 8223.983
n = 114
Левая часть: Σ n_i (u_i+1)^4 = 8223.983
Правая часть: Σ n_i u_i^4 + 4Σ n_i u_i^3 + 6Σ n_i u_i^2 + 4Σ n_i u_i + n =
  3360.510 + 4*762.241 + 6*220.172 + 4*94.868 + 114 =
  3360.510 + 3048.964 + 1321.032 + 379.472 + 114 =
  8223.978

 Равенство выполняется
  Разница: 0.005000


Условные варианты для обоих признаков вычеслены, таблица заполнена, контроль вычислений прошел, что говорит о правильности вычислений.

### 2. Вычислить условные эмпирические моменты $\bar{M_k}^*$ через условные варианты. С помощью условных эмпирических моментов вычислить центральные эмпирические моменты $\bar{m_k}$. Полученные результаты занести в таблицу.

In [7]:
results = {
    'ph': {
        'table': ph_conditional_table,
        'h': ph_h,
        'C': ph_C,
        'n': ph_n,
        'x_mid': ph_data['$\\tilde{x}_i$'].values,
        'freq': ph_data['$m_i$'].values
    },
    'sulphates': {
        'table': sulphates_conditional_table,
        'h': sulphates_h,
        'C': sulphates_C,
        'n': sulphates_n,
        'x_mid': sulphates_data['$\\tilde{x}_i$'].values,
        'freq': sulphates_data['$m_i$'].values
    }
}


In [8]:
def calculate_moments(results, feature_name):
    print(f"ВЫЧИСЛЕНИЕ МОМЕНТОВ ДЛЯ ПРИЗНАКА: {feature_name.upper()}")

    table = results[feature_name]['table']
    n = results[feature_name]['n']
    h = results[feature_name]['h']
    C = results[feature_name]['C']

    sum_ni_ui = table[table['i'] != 'Σ']['$n_i u_i$'].sum()
    sum_ni_ui2 = table[table['i'] != 'Σ']['$n_i u_i^2$'].sum()
    sum_ni_ui3 = table[table['i'] != 'Σ']['$n_i u_i^3$'].sum()
    sum_ni_ui4 = table[table['i'] != 'Σ']['$n_i u_i^4$'].sum()

    # Условные эмпирические моменты M*_k"
    M1_star = sum_ni_ui / n
    M2_star = sum_ni_ui2 / n
    M3_star = sum_ni_ui3 / n
    M4_star = sum_ni_ui4 / n

    # Центральные эмпирические моменты m_k
    # m1 всегда равен 0
    m1 = 0
    m2 = h**2 * (M2_star - M1_star**2)
    m3 = h**3 * (M3_star - 3*M1_star*M2_star + 2*M1_star**3)
    m4 = h**4 * (M4_star - 4*M1_star*M3_star + 6*M1_star**2 * M2_star - 3*M1_star**4)

    moments_table = pd.DataFrame({
        "k": [1, 2, 3, 4],
        "$\\bar{M}^*_k$": [f"{M1_star:.4f}", f"{M2_star:.4f}", f"{M3_star:.4f}", f"{M4_star:.4f}"],
        "$\\bar{m}_k$": [f"{m1}", f"{m2:.4f}", f"{m3:.4f}", f"{m4:.4f}"]
    })

    display(Markdown(moments_table.to_markdown(index=False)))

    moments_results = {
        'M1_star': M1_star,
        'M2_star': M2_star,
        'M3_star': M3_star,
        'M4_star': M4_star,
        'm1': m1,
        'm2': m2,
        'm3': m3,
        'm4': m4,
        'h': h,
        'C': C,
        'n': n
    }

    return moments_results

ph_moments = calculate_moments(results, 'ph')
sulphates_moments = calculate_moments(results, 'sulphates')

ВЫЧИСЛЕНИЕ МОМЕНТОВ ДЛЯ ПРИЗНАКА: PH


|   k |   $\bar{M}^*_k$ |   $\bar{m}_k$ |
|----:|----------------:|--------------:|
|   1 |         -0.4554 |        0      |
|   2 |          1.5201 |        0.0291 |
|   3 |         -1.5498 |        0.0011 |
|   4 |          7.4507 |        0.0031 |

ВЫЧИСЛЕНИЕ МОМЕНТОВ ДЛЯ ПРИЗНАКА: SULPHATES


|   k |   $\bar{M}^*_k$ |   $\bar{m}_k$ |
|----:|----------------:|--------------:|
|   1 |          0.8322 |        0      |
|   2 |          1.9313 |        0.0354 |
|   3 |          6.6863 |        0.0146 |
|   4 |         29.4782 |        0.0113 |

### 3. Вычислить выборочные среднее $\bar{x_в}$ и дисперсию $D_в$ с помощью стандартной формулы и с помощью условных вариант. Убедиться, что результаты совпадают. Вычислить выборочное СКО $\sigma_в$.

In [9]:
def calculate_sample_characteristics(moments_results, raw_data=None, feature_name=""):
    print(f"\n    ВЫЧИСЛЕНИЕ ВЫБОРОЧНЫХ ХАРАКТЕРИСТИК ДЛЯ ПРИЗНАКА: {feature_name}\n")

    C = moments_results['C']
    h = moments_results['h']
    M1_star = moments_results['M1_star']
    m2 = moments_results['m2']
    n = moments_results['n']

    print("Вычисление через условные варианты")
    x_bar_cond = C + h * M1_star
    D_cond = m2
    sigma_cond = np.sqrt(D_cond)

    print(f"    Выборочное среднее через условные варианты: {x_bar_cond:.4f}")
    print(f"    Выборочная дисперсия через условные варианты: {m2:.4f}")
    print(f"    Выборочное СКО: {sigma_cond:.4f}")

    if raw_data is not None:
        print("Вычисление по стандартной формуле")
        x_bar_std = np.mean(raw_data)
        D_std = np.var(raw_data, ddof=0)
        sigma_std = np.std(raw_data, ddof=0)

        print(f"    Выборочное среднее по стандартной формуле: {x_bar_std:.4f}")
        print(f"    Выборочная дисперсия по стандартной формуле: {D_std:.4f}")
        print(f"    Выборочное СКО по стандартной формуле: {sigma_std:.4f}")

        print("\n   СРАВНЕНИЕ РЕЗУЛЬТАТОВ\n")
        print("Сравнение выборочного среднего:")
        diff_mean = abs(x_bar_cond - x_bar_std)
        if diff_mean < 0.01:
            print(f"  РЕЗУЛЬТАТЫ СОВПАДАЮТ (разница: {diff_mean:.6f})")
        else:
            print(f"  РАСХОЖДЕНИЕ (разница: {diff_mean:.6f})")

        print("Сравнение дисперсии:")
        diff_var = abs(D_cond - D_std)
        if diff_var < 0.01:
            print(f"  РЕЗУЛЬТАТЫ СОВПАДАЮТ (разница: {diff_var:.6f})")
        else:
            print(f"  РАСХОЖДЕНИЕ (разница: {diff_var:.6f})")

        print("Сравнение СКО:")
        diff_std = abs(sigma_cond - sigma_std)
        if diff_std < 0.01:
            print(f"  РЕЗУЛЬТАТЫ СОВПАДАЮТ (разница: {diff_std:.6f})")
        else:
            print(f"  РАСХОЖДЕНИЕ (разница: {diff_std:.6f})")

    # Возвращаем результаты
    return {
        'x_bar': x_bar_cond,
        'D': D_cond,
        'sigma': sigma_cond
    }
raw_ph = np.array(data["pH"])
raw_sulphates = np.array(data["sulphates"])

ph_results = calculate_sample_characteristics(ph_moments, raw_ph, "pH")
sulphates_results = calculate_sample_characteristics(sulphates_moments, raw_sulphates, "SULPHATES")


    ВЫЧИСЛЕНИЕ ВЫБОРОЧНЫХ ХАРАКТЕРИСТИК ДЛЯ ПРИЗНАКА: pH

Вычисление через условные варианты
    Выборочное среднее через условные варианты: 3.3121
    Выборочная дисперсия через условные варианты: 0.0291
    Выборочное СКО: 0.1707
Вычисление по стандартной формуле
    Выборочное среднее по стандартной формуле: 3.3077
    Выборочная дисперсия по стандартной формуле: 0.0314
    Выборочное СКО по стандартной формуле: 0.1771

   СРАВНЕНИЕ РЕЗУЛЬТАТОВ

Сравнение выборочного среднего:
  РЕЗУЛЬТАТЫ СОВПАДАЮТ (разница: 0.004426)
Сравнение дисперсии:
  РЕЗУЛЬТАТЫ СОВПАДАЮТ (разница: 0.002238)
Сравнение СКО:
  РЕЗУЛЬТАТЫ СОВПАДАЮТ (разница: 0.006434)

    ВЫЧИСЛЕНИЕ ВЫБОРОЧНЫХ ХАРАКТЕРИСТИК ДЛЯ ПРИЗНАКА: SULPHATES

Вычисление через условные варианты
    Выборочное среднее через условные варианты: 0.6646
    Выборочная дисперсия через условные варианты: 0.0354
    Выборочное СКО: 0.1881
Вычисление по стандартной формуле
    Выборочное среднее по стандартной формуле: 0.6601
    Выборочная диспер

### 4. Вычислить исправленную выборочную дисперсию $s^2$ и исправленное СКО $s$. Сравнить данные оценки с смещёнными оценками дисперсии и СКО. Сделать выводы.

In [15]:
def calculate_corrected_variance(sample_results, raw_data=None, feature_name=""):
    print(f"\nПризнак: {feature_name}\n")

    D_biased = sample_results['D']   # смещённая дисперсия
    sigma_biased = sample_results['sigma']
    n = len(raw_data)

    # Исправленная дисперсия
    s2 = (n / (n - 1)) * D_biased

    # Исправленное СКО
    s = np.sqrt(s2)

    print(f"Объём выборки = {n}")
    print(f"Смещённая дисперсия = {D_biased:.6f}")
    print(f"Исправленная дисперсия = {s2:.6f}")
    print()
    print(f"Смещённое СКО = {sigma_biased:.6f}")
    print(f"Исправленное СКО = {s:.6f}")

    print("\nСРАВНЕНИЕ:")
    print(f"Разница дисперсий: {s2 - D_biased:.6f}")
    print(f"Разница СКО: {s - sigma_biased:.6f}")

    return {
        's2_corrected': s2,
        's_corrected': s
    }
ph_corrected = calculate_corrected_variance(ph_results, raw_ph, "pH")
sulphates_corrected = calculate_corrected_variance(sulphates_results, raw_sulphates, "SULPHATES")


Признак: pH

Объём выборки = 114
Смещённая дисперсия = 0.029143
Исправленная дисперсия = 0.029400

Смещённое СКО = 0.170712
Исправленное СКО = 0.171466

СРАВНЕНИЕ:
Разница дисперсий: 0.000258
Разница СКО: 0.000754

Признак: SULPHATES

Объём выборки = 114
Смещённая дисперсия = 0.035382
Исправленная дисперсия = 0.035695

Смещённое СКО = 0.188101
Исправленное СКО = 0.188931

СРАВНЕНИЕ:
Разница дисперсий: 0.000313
Разница СКО: 0.000830


Исходя из сравнения полученных исправленных оценок видно, что исправленные значения $s^2$ и $s$ незначительно больше смещенных, что соответствует теоретическим положениям. Разница между оценками крайне мала, поскольку объём выборки достаточно большой (114 наблюдений). Следовательно, при данном объёме выборки смещённая и исправленная оценки практически совпадают, однако для статистических выводов корректно использовать именно исправленную дисперсию.

### 5. Найти статистическую оценку коэффициентов асимметрии $\bar{A_s}$ и эксцесса $\bar{E}$. Сделать выводы.

In [17]:
def calculate_skewness_kurtosis(moments_results, feature_name=""):
    print(f"\nПризнак: {feature_name}\n")

    m2 = moments_results['m2']
    m3 = moments_results['m3']
    m4 = moments_results['m4']

    # коэффициент асимметрии
    As = m3 / (m2 ** (3/2))

    # коэффициент эксцесса
    E = (m4 / (m2 ** 2)) - 3

    print(f"Центральный момент m2 = {m2:.6f}")
    print(f"Центральный момент m3 = {m3:.6f}")
    print(f"Центральный момент m4 = {m4:.6f}\n")

    print(f"Коэффициент асимметрии As = {As:.6f}")
    print(f"Коэффициент эксцесса E = {E:.6f}")

    return {
        'As': As,
        'E': E
    }
ph_skew_kurt = calculate_skewness_kurtosis(ph_moments, "pH")
sulphates_skew_kurt = calculate_skewness_kurtosis(sulphates_moments, "SULPHATES")


Признак: pH

Центральный момент m2 = 0.029143
Центральный момент m3 = 0.001118
Центральный момент m4 = 0.003150

Коэффициент асимметрии As = 0.224739
Коэффициент эксцесса E = 0.708432

Признак: SULPHATES

Центральный момент m2 = 0.035382
Центральный момент m3 = 0.014564
Центральный момент m4 = 0.011263

Коэффициент асимметрии As = 2.188296
Коэффициент эксцесса E = 5.997069


Исходя из результатов, можно сделать вывод, что оба признака имеют правостороннюю ассиметрию, однако у признака $pH$ отклонение умеренное, а у признака $sulphates$ коэффициент ассиметрии значительно больше нуля, что говорит о сильной правосторонней асимметрии.

Значение коэффициента эксцесса для обоих признаков положительное, что указывает на островершинность распределения, у признака $sulphates$ наиболее резко выраженную. Таким образом, распределение признака $pH$ близко к симметричному, но несколько большей концентрацией значений около среднего, а признака $sulphates$ - существенно отклоняется от нормального с высокой концентрацией значений вблизи среднего при наличии выбросов.

### 6. Для интервального ряда вычислить моду $M_o^*$, медиану $M_e^*$ и коэффициент вариации $V^*$ заданного распределения. Сделать выводы.

In [20]:
def calculate_mode_median_variation(interval_table, sample_results, raw_data, feature_name=""):
    print(f"\nПризнак: {feature_name}\n")

    df = interval_table.iloc[:-1].copy()
    n = len(raw_data)

    intervals = df["$[x_i, x_{i+1})$"].str.extract(r"\[(.*), (.*)\)").astype(float)
    df["L"] = intervals[0]
    df["R"] = intervals[1]

    # шаг интервала
    h = df["R"].iloc[0] - df["L"].iloc[0]

    # мода
    modal_index = df["$m_i$"].idxmax()
    Lm = df.loc[modal_index, "L"]
    fm = df.loc[modal_index, "$m_i$"]

    f_prev = df.loc[modal_index - 1, "$m_i$"] if modal_index > 0 else 0
    f_next = df.loc[modal_index + 1, "$m_i$"] if modal_index < len(df)-1 else 0

    Mo = Lm + h * ((fm - f_prev) / ((fm - f_prev) + (fm - f_next)))

    # медиана
    df["cum_freq"] = df["$m_i$"].cumsum()
    median_index = df[df["cum_freq"] >= n/2].index[0]

    Lmed = df.loc[median_index, "L"]
    F_prev = df.loc[median_index - 1, "cum_freq"] if median_index > 0 else 0
    f_med = df.loc[median_index, "$m_i$"]

    Me = Lmed + h * ((n/2 - F_prev) / f_med)

    # коэффициент вариации
    sigma = sample_results["sigma"]
    x_bar = sample_results["x_bar"]

    V = (sigma / x_bar) * 100

    print(f"Мода Mo* = {Mo:.4f}")
    print(f"Медиана Me* = {Me:.4f}")
    print(f"Коэффициент вариации V* = {V:.2f}%")

    return {
        "mode": Mo,
        "median": Me,
        "variation": V
    }
ph_distribution = calculate_mode_median_variation(ph_table, ph_results, raw_ph, "pH")
sulphates_distribution = calculate_mode_median_variation(sulphates_table, sulphates_results, raw_sulphates, "SULPHATES")


Признак: pH

Мода Mo* = 3.3245
Медиана Me* = 3.3173
Коэффициент вариации V* = 5.15%

Признак: SULPHATES

Мода Mo* = 0.5808
Медиана Me* = 0.6262
Коэффициент вариации V* = 28.30%


Мода и медиана признака $pH$ близки друг к другу, что свидетельствует о близости распределения к симметричному. Коэффициент вариации равен $5.15\% $, что меньше $10\% $. Это означает слабую вариацию признака и высокую однородность совокупности. Следовательно, значения $pH$ мало отклоняются от среднего уровня и распределены достаточно равномерно.

У признака $sulphates$ мода меньше медианы, что подтверждает наличие правосторонней асимметрии распределения. Коэффициент вариации составляет $28.30\%$, что свидетельствует о высокой вариации признака и неоднородности совокупности. Значения $sulphates$ имеют значительный разброс относительно среднего, что подтверждает ранее выявленную сильную асимметрию и наличие высоких значений.

### Вывод

В ходе выполнения практической работы выполнено нахождение точечных статистических оценок параметров распределения на основе выборочных данных с использованием метода моментов и условных вариантов.

Полученные моменты позволили определить числовые характеристики распределения, такие как дисперсия, среднеквадратичное отклонение, асимметрия и эксцесс, а также проверили корректность вычислений с использованием условных вариантов.

Выборочные характеристики рассчитаны двумя способами: стандартной формулой и через условные варианты. Для обоих признаков результаты совпали, что подтверждает корректность метода. Использование условных вариантов подтвердило совпадение расчетов и дало возможность работать с интервальными рядами более эффективно.

Для оценки несмещённости дисперсии и СКО рассчитаны исправленные значения. Исправленная дисперсия и СКО оказались немного больше смещённых для обоих признаков. Разница между смещёнными и исправленными оценками крайне мала, что объясняется достаточно большим объёмом выборки.

Анализ формы распределения показал различия между признаками. Для признака $pH$ коэффициент асимметрии $A_s=0.22$ указывает на умеренную правостороннюю асимметрию, а коэффициент эксцесса $E=0.71$ свидетельствует о слегка островершинной форме распределения. Для признака $sulphates$ асимметрия выражена гораздо сильнее $A_s=2.19$, а коэффициент эксцесса $E=5.99$ указывает на резко выраженную островершинность. Это подтверждает, что распределение $sulphates$ значительно отклоняется от нормального закона, в отличие от распределения $pH$, которое близко к симметричному.

Далее рассчитаны мода, медиана и коэффициент вариации на основе интервального ряда. Для  признака $pH$ коэффициент вариации $V^∗=5.15\%$ показывают низкую вариацию и высокую однородность выборки. Для признака $sulphates$ коэффициент вариации $V^∗=28.30\%$ указывает на высокую вариацию и неоднородность данных, что соответствует выявленной сильной асимметрии и островершинности распределения.

Проведённая работа позволила получить полное представление о распределении исследуемых признаков, определить основные числовые характеристики и сделать выводы о форме, однородности и разбросе данных выборки.